In [5]:
import nest_asyncio
nest_asyncio.apply()

In [ ]:
import os

# LM Studio server from your screenshot
LMSTUDIO_BASE_URL = ""z
LMSTUDIO_API_KEY  = "lm-studio"   # any dummy string is fine

# Two models you have loaded
SMALL_MODEL_ID = "qwen2.5-3b-instruct"
BIG_VLM_ID     = "qwen/qwen3-vl-4b"

# Make LM Studio look like a vLLM server to cascadeflow
os.environ["VLLM_BASE_URL"] = LMSTUDIO_BASE_URL
os.environ["VLLM_API_KEY"]  = LMSTUDIO_API_KEY

# optional: keep model ids in env for convenience
os.environ["VLLM_DRAFT_MODEL"]    = SMALL_MODEL_ID
os.environ["VLLM_VERIFIER_MODEL"] = BIG_VLM_ID

LMSTUDIO_BASE_URL, SMALL_MODEL_ID, BIG_VLM_ID


('http://10.137.22.177:9000/v1', 'qwen2.5-3b-instruct', 'qwen/qwen3-vl-4b')

In [11]:
import requests

health_url = LMSTUDIO_BASE_URL.replace("/v1", "") + "/v1/models"
print("Checking LM Studio at:", health_url)

resp = requests.get(health_url, headers={"Authorization": f"Bearer {LMSTUDIO_API_KEY}"})
print("Status:", resp.status_code)
print("Response JSON:")
resp.json()


Checking LM Studio at: http://10.137.22.177:9000/v1/models
Status: 200
Response JSON:


{'data': [{'id': 'qwen2.5-3b-instruct',
   'object': 'model',
   'owned_by': 'organization_owner'},
  {'id': 'qwen/qwen3-vl-4b',
   'object': 'model',
   'owned_by': 'organization_owner'},
  {'id': 'google/gemma-3-12b',
   'object': 'model',
   'owned_by': 'organization_owner'},
  {'id': 'qwen/qwen3-vl-8b',
   'object': 'model',
   'owned_by': 'organization_owner'},
  {'id': 'openai/gpt-oss-20b',
   'object': 'model',
   'owned_by': 'organization_owner'},
  {'id': 'meta-llama-3-8b-instruct',
   'object': 'model',
   'owned_by': 'organization_owner'},
  {'id': 'google/gemma-3n-e4b',
   'object': 'model',
   'owned_by': 'organization_owner'},
  {'id': 'deepseek/deepseek-r1-0528-qwen3-8b',
   'object': 'model',
   'owned_by': 'organization_owner'},
  {'id': 'text-embedding-nomic-embed-text-v1.5',
   'object': 'model',
   'owned_by': 'organization_owner'}],
 'object': 'list'}

In [12]:
from cascadeflow import CascadeAgent, ModelConfig, QualityConfig

def create_lmstudio_vlm_agent() -> CascadeAgent:
    """
    Two-tier cascade using cascadeflow's vLLM provider,
    with LM Studio as the OpenAI-compatible backend.
    Tier 1: qwen2.5-3b-instruct  (fast / cheap)
    Tier 2: qwen/qwen3-vl-4b     (vision-enabled, stronger)
    """

    base_url = os.environ["VLLM_BASE_URL"]
    api_key  = os.environ["VLLM_API_KEY"]

    small_model = os.environ.get("VLLM_DRAFT_MODEL", "qwen2.5-3b-instruct")
    big_model   = os.environ.get("VLLM_VERIFIER_MODEL", "qwen/qwen3-vl-4b")

    models = [
        # Tier 1 – draft / cheap
        ModelConfig(
            name=small_model,
            provider="vllm",
            cost=0.0,                 # local → $0
            base_url=base_url,
            api_key=api_key,
            quality_threshold=0.7,
            metadata={"tier": 1, "label": "draft"},
        ),
        # Tier 2 – verifier / strong VLM
        ModelConfig(
            name=big_model,
            provider="vllm",
            cost=0.0,
            base_url=base_url,
            api_key=api_key,
            quality_threshold=0.95,
            metadata={"tier": 2, "label": "verifier"},
        ),
    ]

    quality_config = QualityConfig(
        confidence_thresholds={
            "trivial": 0.65,
            "simple": 0.60,
            "moderate": 0.55,
            "hard": 0.50,
            "expert": 0.45,
        },
        enable_adaptive=True,
    )

    agent = CascadeAgent(
        models=models,
        quality_config=quality_config,
        enable_cascade=True,
        verbose=True,
    )
    return agent


# def create_lmstudio_vlm_agent() -> CascadeAgent:
#     """
#     Two-tier cascade over LM Studio, accessed via OpenAI-compatible provider.
#     Tier 1: qwen2.5-3b-instruct   (fast / cheap)
#     Tier 2: qwen/qwen3-vl-4b      (vision-enabled, stronger)
#     """

#     small_model = os.environ.get("LMSTUDIO_SMALL_MODEL", "qwen2.5-3b-instruct")
#     big_model   = os.environ.get("LMSTUDIO_BIG_MODEL", "qwen/qwen3-vl-4b")

#     models = [
#         ModelConfig(
#             name=small_model,
#             provider="openai",      # <── changed
#             cost=0.0,
#             quality_threshold=0.7,
#             metadata={"tier": 1, "label": "draft"},
#         ),
#         ModelConfig(
#             name=big_model,
#             provider="openai",      # <── changed
#             cost=0.0,
#             quality_threshold=0.95,
#             metadata={"tier": 2, "label": "verifier"},
#         ),
#     ]

#     quality_config = QualityConfig(
#         confidence_thresholds={
#             "trivial": 0.65,
#             "simple": 0.60,
#             "moderate": 0.55,
#             "hard": 0.50,
#             "expert": 0.45,
#         },
#         enable_adaptive=True,
#     )

#     agent = CascadeAgent(
#         models=models,
#         quality_config=quality_config,
#         enable_cascade=True,
#         verbose=True,
#     )
#     return agent



In [13]:
agent = create_lmstudio_vlm_agent()
agent

INFO:cascadeflow.routing.pre_router:PreRouter initialized:
  Cascade enabled: True
  Cascade complexities: ['trivial', 'simple', 'moderate']
  Direct complexities: ['hard', 'expert']
INFO:cascadeflow.routing.tool_router:ToolRouter initialized: 2/2 models support tools
INFO:cascadeflow.routing.tool_router:Tool-capable models: qwen2.5-3b-instruct, qwen/qwen3-vl-4b
INFO:cascadeflow.telemetry.collector:MetricsCollector initialized: max_recent=100, verbose=True
INFO:cascadeflow.telemetry.cost_calculator:CostCalculator initialized:
  Drafter: qwen2.5-3b-instruct ($0.0/1K tokens)
  Verifier: qwen/qwen3-vl-4b ($0.0/1K tokens)
INFO:cascadeflow.providers.base:LiteLLM detected - using accurate pricing for VLLMProvider
INFO:cascadeflow.providers.base:LiteLLM detected - using accurate pricing for VLLMProvider
INFO:cascadeflow.telemetry.cost_calculator:CostCalculator initialized:
  Drafter: qwen2.5-3b-instruct ($0.0/1K tokens)
  Verifier: qwen/qwen3-vl-4b ($0.0/1K tokens)
INFO:cascadeflow.core.casca

In [17]:
agent = create_lmstudio_vlm_agent()

INFO:cascadeflow.routing.pre_router:PreRouter initialized:
  Cascade enabled: True
  Cascade complexities: ['trivial', 'simple', 'moderate']
  Direct complexities: ['hard', 'expert']
INFO:cascadeflow.routing.tool_router:ToolRouter initialized: 2/2 models support tools
INFO:cascadeflow.routing.tool_router:Tool-capable models: qwen2.5-3b-instruct, qwen/qwen3-vl-4b
INFO:cascadeflow.telemetry.collector:MetricsCollector initialized: max_recent=100, verbose=True
INFO:cascadeflow.telemetry.cost_calculator:CostCalculator initialized:
  Drafter: qwen2.5-3b-instruct ($0.0/1K tokens)
  Verifier: qwen/qwen3-vl-4b ($0.0/1K tokens)
INFO:cascadeflow.providers.base:LiteLLM detected - using accurate pricing for VLLMProvider
INFO:cascadeflow.providers.base:LiteLLM detected - using accurate pricing for VLLMProvider
INFO:cascadeflow.telemetry.cost_calculator:CostCalculator initialized:
  Drafter: qwen2.5-3b-instruct ($0.0/1K tokens)
  Verifier: qwen/qwen3-vl-4b ($0.0/1K tokens)
INFO:cascadeflow.core.casca

In [ ]:
import asyncio
# async def quick_test():
def quick_test():
    
    # res = await agent.run("Say: routed via vLLM provider.", max_tokens=32)
    res = agent.run("Say: routed via vLLM provider.", max_tokens=32)
    # print("Model used:", res.model_used)
    # print("Content:", res.content)
    print("Response:", res)
    return res
# asyncio.run(quick_test())
res = quick_test()    

INFO:cascadeflow.agent:Query complexity: simple (confidence: 0.60)
INFO:cascadeflow.agent:Routing to cascade: qwen2.5-3b-instruct → qwen/qwen3-vl-4b
INFO:cascadeflow.core.cascade:📝 TEXT PATH: No tools
INFO:cascadeflow.core.cascade:Starting cascade: qwen2.5-3b-instruct → qwen/qwen3-vl-4b
ERROR:cascadeflow.providers.base:VLLM: ✗ Not retrying unknown on attempt 1/3: unknown async library, or not in async context
ERROR:cascadeflow.core.cascade:Drafter error: unknown async library, or not in async context
Traceback (most recent call last):
  File "/Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Which_VLM_Router/code_base/cascadeflow/cascadeflow_env/lib/python3.14/site-packages/httpcore/_async/connection_pool.py", line 228, in handle_async_request
    closing = self._assign_requests_to_connections()
  File "/Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Which_VLM_Router/code_base/cascadeflow/cascadeflow_env/lib/python3.14/site-pa

[Complexity: simple (confidence: 0.60)]
[Detection time: 9.7ms]
[PreRouter] Say: routed via vLLM provider.... → cascade
           Complexity: simple (conf: 0.60)
           Reason: simple query suitable for cascade optimization
[Routing: CASCADE]
[Reason: simple query suitable for cascade optimization]
[Confidence: 0.60]
[Draft generation: 10.9ms]
[Quality check: 0.0ms]
[Verifier generation: 2.3ms]
[Cascade overhead: 10.9ms]
[Total latency: 25.2ms]
Model used: qwen2.5-3b-instruct+qwen/qwen3-vl-4b
Content: 


In [21]:
print("Model used:", res.model_used)
print("Content:", res.content)

AttributeError: 'coroutine' object has no attribute 'model_used'

In [8]:
from datasets import load_dataset

# Use a small slice first so it runs quickly
cauldron_config = "ai2d"  # change if you want another config
split = "train[:20]" # or "train[:N]"

ds = load_dataset("HuggingFaceM4/the_cauldron", cauldron_config, split=split)
len(ds), ds.column_names


INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/HuggingFaceM4/the_cauldron/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/HuggingFaceM4/the_cauldron/847a98a779b1652d65111daf20c972dfcd333605/README.md "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/HuggingFaceM4/the_cauldron/resolve/847a98a779b1652d65111daf20c972dfcd333605/the_cauldron.py "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/HuggingFaceM4/the_cauldron/HuggingFaceM4/the_cauldron.py "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/HuggingFaceM4/the_cauldron/resolve/847a98a779b1652d65111daf20c972dfcd333605/.huggingface.yaml "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://datasets-server.huggingface.co/info?dataset=HuggingFaceM4/the_cauldron "HTTP/1.1 200 OK"
INFO:httpx:HT

(20, ['images', 'texts'])

In [ ]:
sample = ds[0]
sample


In [ ]:
from typing import Dict, Any

def cauldron_to_prompt(sample: Dict[str, Any]) -> str:
    """
    Simple adapter: build a plain-text prompt for the router.
    Uses the last text turn as the question (AI2D style).
    """
    texts = sample.get("texts", [])
    if isinstance(texts, list) and len(texts) > 0:
        # Often last text is the actual question
        last = texts[-1]
        if isinstance(last, dict):
            question = last.get("content", "") or last.get("text", "")
        else:
            question = str(last)
    else:
        question = sample.get("question", "")

    return (
        "You are a helpful model answering questions about an image."
        "\n(The image is not shown in this first experiment; answer from the text only.)\n\n"
        f"Question: {question}\n"
        "Answer clearly and concisely."
    )

print(cauldron_to_prompt(sample))


In [ ]:
import asyncio
import time
import pandas as pd

async def route_one_sample(agent: CascadeAgent, sample: Dict[str, Any], idx: int):
    prompt = cauldron_to_prompt(sample)
    start = time.time()
    result = await agent.run(prompt, max_tokens=256)
    latency_ms = (time.time() - start) * 1000
    
    model_used   = result.model_used or ""
    cascaded     = getattr(result, "cascaded", False)
    quality      = getattr(result, "quality_score", None)
    content      = result.content
    
    # Decide tier from name
    if SMALL_MODEL_ID in model_used:
        tier = 1
    elif BIG_VLM_ID in model_used:
        tier = 2
    else:
        tier = None
    
    print(f"Sample {idx}: model={model_used} (tier {tier}), "
          f"cascaded={cascaded}, quality={quality}, latency={latency_ms:.0f} ms")
    
    return {
        "index": idx,
        "model_used": model_used,
        "tier": tier,
        "cascaded": cascaded,
        "quality": quality,
        "latency_ms": latency_ms,
        "answer_preview": content[:200].replace("\n", " "),
    }

async def run_routing_experiment(dataset, n_samples: int = 10):
    agent = create_lmstudio_vlm_agent()
    rows = []
    for i in range(min(n_samples, len(dataset))):
        rows.append(await route_one_sample(agent, dataset[i], i))
    return pd.DataFrame(rows)

df_results = asyncio.run(run_routing_experiment(ds, n_samples=10))
df_results


In [ ]:
df_results["tier"].value_counts(normalize=True) * 100


In [ ]:
df_results.groupby("tier")[["latency_ms"]].mean()


In [ ]:
df_results.head()


In [ ]:
def cauldron_to_messages_with_image(sample: Dict[str, Any], image_root: str) -> list[Dict[str, Any]]:
    """
    Build an OpenAI-style multi-modal messages payload for LM Studio.
    Assumes you have saved Cauldron images locally under `image_root`.
    """
    # This part depends on how you've stored the images locally.
    # Example if you saved them as '<image_root>/<sample_id>.png':
    #   image_path = os.path.join(image_root, sample["id"] + ".png")
    # Adjust to your own filename convention.
    
    # TODO: adapt to your real image path mapping
    image_rel = sample["images"][0]  # or sample["images"][0]["path"], etc.
    image_path = os.path.join(image_root, image_rel)
    
    texts = sample.get("texts", [])
    last = texts[-1]
    if isinstance(last, dict):
        question = last.get("content", "") or last.get("text", "")
    else:
        question = str(last)

    content = [
        {"type": "text", "text": "Answer the question based on the image."},
        {"type": "image_url", "image_url": {"url": f"file://{image_path}"}},
        {"type": "text", "text": f"Question: {question}"},
    ]
    return [{"role": "user", "content": content}]


In [ ]:
result = await agent.run(messages=cauldron_to_messages_with_image(sample, image_root="..."), max_tokens=256)


In [ ]:
output_path = "lmstudio_cauldron_routing_results.csv"
df_results.to_csv(output_path, index=False)
output_path


In [25]:
import os
import requests

# -----------------------------
# LM Studio configuration
# -----------------------------
LMSTUDIO_BASE_URL = "http://10.137.22.177:9000/v1"
LMSTUDIO_API_KEY  = "lm-studio"   # LM Studio ignores this but it must be a non-empty string

# Tell any OpenAI-compatible client (including cascadeflow's OpenAI provider)
# to talk to LM Studio instead of api.openai.com
os.environ["OPENAI_API_KEY"] = LMSTUDIO_API_KEY
os.environ["OPENAI_BASE_URL"] = LMSTUDIO_BASE_URL

# Optional: quick health check against LM Studio
health_url = LMSTUDIO_BASE_URL.replace("/v1", "") + "/v1/models"
print("Checking LM Studio at:", health_url)

resp = requests.get(health_url, headers={"Authorization": f"Bearer {LMSTUDIO_API_KEY}"})
print("Status:", resp.status_code)
print("Response JSON:", resp.json())


Checking LM Studio at: http://10.137.22.177:9000/v1/models
Status: 200
Response JSON: {'data': [{'id': 'qwen2.5-3b-instruct', 'object': 'model', 'owned_by': 'organization_owner'}, {'id': 'qwen/qwen3-vl-4b', 'object': 'model', 'owned_by': 'organization_owner'}, {'id': 'google/gemma-3-12b', 'object': 'model', 'owned_by': 'organization_owner'}, {'id': 'qwen/qwen3-vl-8b', 'object': 'model', 'owned_by': 'organization_owner'}, {'id': 'openai/gpt-oss-20b', 'object': 'model', 'owned_by': 'organization_owner'}, {'id': 'meta-llama-3-8b-instruct', 'object': 'model', 'owned_by': 'organization_owner'}, {'id': 'google/gemma-3n-e4b', 'object': 'model', 'owned_by': 'organization_owner'}, {'id': 'deepseek/deepseek-r1-0528-qwen3-8b', 'object': 'model', 'owned_by': 'organization_owner'}, {'id': 'text-embedding-nomic-embed-text-v1.5', 'object': 'model', 'owned_by': 'organization_owner'}], 'object': 'list'}


In [27]:
from cascadeflow import CascadeAgent, ModelConfig, QualityConfig

def create_lmstudio_vlm_agent() -> CascadeAgent:
    """
    Two-tier cascade over LM Studio, using the OpenAI-compatible provider.

    Tier 1: qwen2.5-3b-instruct   (fast / cheap)
    Tier 2: qwen/qwen3-vl-4b      (vision-enabled, stronger)
    """

    small_model = "qwen2.5-3b-instruct"
    big_model   = "qwen/qwen3-vl-4b"

    models = [
        # Tier 1 – draft / cheap
        ModelConfig(
            name=small_model,
            provider="openai",      # <── IMPORTANT: use openai provider, not vllm
            cost=0.0,
            quality_threshold=0.7,
            metadata={"tier": 1, "label": "draft"},
        ),
        # Tier 2 – verifier / strong VLM
        ModelConfig(
            name=big_model,
            provider="openai",      # <── also openai
            cost=0.0,
            quality_threshold=0.95,
            metadata={"tier": 2, "label": "verifier"},
        ),
    ]

    quality_config = QualityConfig(
        confidence_thresholds={
            "trivial": 0.65,
            "simple": 0.60,
            "moderate": 0.55,
            "hard": 0.50,
            "expert": 0.45,
        },
        enable_adaptive=True,
    )

    agent = CascadeAgent(
        models=models,
        quality_config=quality_config,
        enable_cascade=True,
        verbose=True,
    )
    return agent




In [28]:
agent = create_lmstudio_vlm_agent()
agent

INFO:cascadeflow.routing.pre_router:PreRouter initialized:
  Cascade enabled: True
  Cascade complexities: ['trivial', 'simple', 'moderate']
  Direct complexities: ['hard', 'expert']
INFO:cascadeflow.routing.tool_router:ToolRouter initialized: 2/2 models support tools
INFO:cascadeflow.routing.tool_router:Tool-capable models: qwen2.5-3b-instruct, qwen/qwen3-vl-4b
INFO:cascadeflow.telemetry.collector:MetricsCollector initialized: max_recent=100, verbose=True
INFO:cascadeflow.telemetry.cost_calculator:CostCalculator initialized:
  Drafter: qwen2.5-3b-instruct ($0.0/1K tokens)
  Verifier: qwen/qwen3-vl-4b ($0.0/1K tokens)
INFO:cascadeflow.providers.base:LiteLLM detected - using accurate pricing for OpenAIProvider
INFO:cascadeflow.providers.base:LiteLLM detected - using accurate pricing for OpenAIProvider
INFO:cascadeflow.telemetry.cost_calculator:CostCalculator initialized:
  Drafter: qwen2.5-3b-instruct ($0.0/1K tokens)
  Verifier: qwen/qwen3-vl-4b ($0.0/1K tokens)
INFO:cascadeflow.core.c

In [32]:
# Simple test query

res = await agent.run("Say: Hello from LM Studio via CascadeFlow.", max_tokens=32)

print("MODEL USED:", res.model_used)
print("CONTENT:", repr(res.content))
print("CASCADED:", res.cascaded)
print("DRAFT ACCEPTED:", getattr(res, "draft_accepted", None))
print("COMPLEXITY:", getattr(res, "complexity", None))
print("TOTAL COST:", res.total_cost)
print("LATENCY (ms):", res.latency_ms)


INFO:cascadeflow.agent:Query complexity: simple (confidence: 0.60)
INFO:cascadeflow.agent:Routing to cascade: qwen2.5-3b-instruct → qwen/qwen3-vl-4b
INFO:cascadeflow.core.cascade:📝 TEXT PATH: No tools
INFO:cascadeflow.core.cascade:Starting cascade: qwen2.5-3b-instruct → qwen/qwen3-vl-4b
ERROR:cascadeflow.providers.base:OpenAI: ✗ Not retrying unknown on attempt 1/3: unknown async library, or not in async context
ERROR:cascadeflow.core.cascade:Drafter error: unknown async library, or not in async context
Traceback (most recent call last):
  File "/Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Which_VLM_Router/code_base/cascadeflow/cascadeflow_env/lib/python3.14/site-packages/httpcore/_async/connection_pool.py", line 228, in handle_async_request
    closing = self._assign_requests_to_connections()
  File "/Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Which_VLM_Router/code_base/cascadeflow/cascadeflow_env/lib/python3.14/site-

[Complexity: simple (confidence: 0.60)]
[Detection time: 0.5ms]
[PreRouter] Say: Hello from LM Studio via CascadeFlow.... → cascade
           Complexity: simple (conf: 0.60)
           Reason: simple query suitable for cascade optimization
[Routing: CASCADE]
[Reason: simple query suitable for cascade optimization]
[Confidence: 0.60]
[Draft generation: 2.4ms]
[Quality check: 0.0ms]
[Verifier generation: 2.3ms]
[Cascade overhead: 2.4ms]
[Total latency: 7.3ms]
MODEL USED: qwen2.5-3b-instruct+qwen/qwen3-vl-4b
CONTENT: ''
CASCADED: True
DRAFT ACCEPTED: False
COMPLEXITY: simple
TOTAL COST: 0.0
LATENCY (ms): 7.340908050537109
